# 1. Load the data and explore the SummarizedExperiment

We work with `intracell_raw_se`, a `SummarizedExperiment` shipped with
MetaProViz. It contains intracellular metabolomics from a kidney-cancer
study (Metabolomics Workbench `ST002224`): three ccRCC cell lines (786-O,
786-M1A, 786-M2A) and a non-cancerous proximal-tubule control (HK2), with
pool samples used for QC.

Since v3.99.10 (Oct 2025) MetaProViz uses `SummarizedExperiment` (SE) as
its canonical container — the same object Bioconductor's omics packages
expect. Each row is a metabolite; each column is a sample. Sample
annotations live in `colData(se)`, feature annotations in `rowData(se)`,
the actual measurements in `assays(se)`.

In [ ]:
source(file.path(here::here(), "scripts/R/_utils.R"))
library(MetaProViz)
library(OmnipathR)

In [ ]:
data(intracell_raw_se)
intracell_raw_se

## What's in the SE?

In [ ]:
SummarizedExperiment::assayNames(intracell_raw_se)
dim(intracell_raw_se)

In [ ]:
# Sample metadata: condition, biological replicate, analytical replicate.
colData(intracell_raw_se) |> as.data.frame() |> head()

In [ ]:
# Feature metadata: only metabolite names at this stage.
rowData(intracell_raw_se) |> as.data.frame() |> head()

In [ ]:
# Raw peak values (rows = metabolites, columns = samples).
assay(intracell_raw_se)[1:6, 1:6]

## Pre-processing

`processing()` (was `PreProcessing()` before v3.0.1) takes one SE in,
returns one SE out. By default it does:

- **feature filtering** — drops metabolites detected in <80% of samples
  per condition (`featurefilt = "Modified"`)
- **TIC normalisation** — scales each sample by its total ion count
- **half-minimum imputation** of missing values
- **outlier detection** via Hotelling's T² (records outliers in
  `colData()`)

All choices are echoed to the log; QC plots are returned in the result
list and also saved under `MetaProViz_Results/` inside the working dir.

In [ ]:
processed <- processing(
    data = intracell_raw_se,
    metadata_info = c(
        Conditions = "Conditions",
        Biological_Replicates = "Biological_Replicate"
    ),
    featurefilt = "Modified",
    cutoff_featurefilt = 0.8,
    tic = TRUE,
    mvi = TRUE,
    save_plot = "svg",
    save_table = "csv",
    print_plot = FALSE,
    path = mp_results_dir("01_processing")
)

In [ ]:
# The processed SE: same shape, with imputed/normalised values and outliers
# tagged in colData.
processed_se <- processed$SE
processed_se

In [ ]:
SummarizedExperiment::assayNames(processed_se)
table(processed_se$Outliers)

We drop pool samples (used for QC) and any flagged outliers before the
differential analysis in script 02.

In [ ]:
keep <- processed_se$Conditions != "Pool" & processed_se$Outliers == "no"
clean_se <- processed_se[, keep]
clean_se

In [ ]:
saveRDS(clean_se, file.path(mp_results_dir(), "intracell_clean_se.rds"))

**Recap.** We loaded a SE, ran one `processing()` call, dropped pools and
outliers, and saved the cleaned object. In script 02 we'll compute
differential abundance with `dma()`.

# 2. Differential metabolite analysis

We compare each ccRCC cell line (786-O, 786-M1A, 786-M2A) to HK2 with
`dma()` — MetaProViz's `D`ifferential `M`etabolite `A`nalysis routine.
`dma()` (formerly `DMA()`) consumes a `SummarizedExperiment` and returns
one SE per pairwise contrast plus diagnostic plots.

In [ ]:
source(file.path(here::here(), "scripts/R/_utils.R"))
library(MetaProViz)

clean_se <- readRDS(file.path(mp_results_dir(), "intracell_clean_se.rds"))
clean_se

## Run `dma()`

`metadata_info` is the new snake_case parameter (was `MetadataInfo`). To
do all-vs-HK2 in one call, we set `Numerator = NULL` and
`Denominator = "HK2"` — every other condition becomes a contrast against
HK2. Stats engine is limma (`pval = "lmFit"`); FDR adjustment is BH.

In [ ]:
dma_results <- dma(
    data = clean_se,
    metadata_info = c(
        Conditions = "Conditions",
        Numerator = NULL,
        Denominator = "HK2"
    ),
    pval = "lmFit",
    padj = "fdr",
    save_plot = "svg",
    save_table = "csv",
    print_plot = FALSE,
    path = mp_results_dir("02_dma")
)

names(dma_results)

Each list element is a contrast. Results live as new assays on the SE
along with a tidy results table.

In [ ]:
contrast <- "786-M1A_vs_HK2"
res_se <- dma_results[[contrast]]$SE
res_se

In [ ]:
SummarizedExperiment::assayNames(res_se)

In [ ]:
# The rectangular results table also comes back ready to plot.
dma_table <- dma_results[[contrast]]$dma
head(dma_table)

## Volcano plot

`viz_volcano()` (was `VizVolcano()`) accepts the rectangular dma table
and uses sensible defaults; we only label a handful of metabolites.

In [ ]:
viz_volcano(
    plot_types = "Standard",
    data = dma_table |> tibble::column_to_rownames("Metabolite"),
    cutoff_x = 0.5,
    cutoff_y = 0.05,
    select_label = c("citrate", "succinate", "lactate", "alpha-ketoglutarate"),
    plot_name = contrast,
    subtitle = "Differential metabolites: 786-M1A vs HK2",
    save_plot = "svg",
    print_plot = TRUE,
    path = mp_results_dir("02_dma")
)

## Heatmap of the top differential metabolites

`viz_heatmap()` is the function where the X11 font error used to bite on
Ubuntu LTS. The repo's `.Rprofile` forces the cairo device, so this just
works.

In [ ]:
top_metabolites <- dma_table |>
    dplyr::arrange(p.adj) |>
    dplyr::slice_head(n = 30) |>
    dplyr::pull(Metabolite)

viz_heatmap(
    data = clean_se[top_metabolites, ],
    metadata_info = c(color = "Conditions"),
    plot_name = "Top 30 differential metabolites",
    save_plot = "svg",
    print_plot = TRUE,
    path = mp_results_dir("02_dma")
)

## Sample structure: PCA

A quick PCA confirms cell-line separation along PC1.

In [ ]:
viz_pca(
    data = clean_se,
    metadata_info = c(color = "Conditions"),
    plot_name = "PCA of cleaned intracellular metabolomics",
    save_plot = "svg",
    print_plot = TRUE,
    path = mp_results_dir("02_dma")
)

In [ ]:
saveRDS(dma_results, file.path(mp_results_dir(), "dma_results.rds"))

**Recap.** `dma()` produced three contrasts (`*_vs_HK2`). Volcano +
heatmap + PCA give us a quick read on biology before we move to
pathway-level interpretation in scripts 03-05.

# 3. Metabolite identifier translation

Most prior-knowledge resources speak different ID dialects (HMDB, ChEBI,
KEGG, PubChem, LipidMaps, …). Before we can connect dma results to
pathways, we need to translate metabolite identifiers.

Two complementary tools:

- **OmnipathR** wraps RaMP-DB and its companion mapping infrastructure;
  `translate_ids()` is the general cross-resource translator.
- **MetaProViz** wraps OmnipathR with a tidier, table-in / table-out API
  tuned for metabolomics workflows: `translate_id()` (singular),
  `equivalent_id()`, and `count_id()`.

In 2025 we showed 16 cells of variations on this. Here we pick three
representative ones and move the rest to `99_appendix_legacy.R`.

In [ ]:
source(file.path(here::here(), "scripts/R/_utils.R"))
library(MetaProViz)
library(OmnipathR)

## What identifier types are available?

In [ ]:
# Resources that OmnipathR can translate between for metabolites.
id_translation_resources(entity = "metabolite") |> head()

## Example A — translate a small KEGG pathway list to HMDB and PubChem

`translate_id()` takes the input column name in `metadata_info`, plus
`from` / `to` IDs.

In [ ]:
kegg_pw <- metsigdb_kegg()
head(kegg_pw)

In [ ]:
kegg_to_hmdb <- translate_id(
    data = kegg_pw,
    metadata_info = c(InputID = "MetaboliteID", grouping_variable = "term"),
    from = "kegg",
    to = c("hmdb", "pubchem")
)

names(kegg_to_hmdb)
head(kegg_to_hmdb$TranslatedDF)

Translation is rarely 1-to-1. The result list also reports mapping
ambiguity, which you should always check before downstream enrichment.

In [ ]:
kegg_to_hmdb$MappingAmbiguity |> head()

## Example B — `equivalent_id()` to enrich a dataset with cross-IDs

Given a table that already has one ID column (here HMDB), expand it
with all known equivalents.

In [ ]:
example_hmdb <- data.frame(
    MetaboliteName = c("citrate", "succinate", "lactate"),
    HMDB = c("HMDB0000094", "HMDB0000254", "HMDB0000190")
)

with_equivalents <- equivalent_id(
    data = example_hmdb,
    metadata_info = c(InputID = "HMDB"),
    from = "hmdb"
)

head(with_equivalents)

## Example C — quick coverage check with `count_id()`

In [ ]:
count_id(with_equivalents, "HMDB")$result

**Recap.** With three concise calls we have the IDs we need to talk to
the prior-knowledge resources in script 04. For more variations
(Biocrates feature tables, multi-comma-separated cells, mapping
ambiguity surgery) see `99_appendix_legacy.R`.

# 4. Metabolomics prior knowledge from OmnipathR

OmnipathR has gained substantial metabolomics coverage in the last year.
The four resources we'll use:

- **MetalinksDB** — metabolite ↔ protein interactions (Saez group)
- **HMDB** — Human Metabolome Database
- **RaMP-DB** — pathway / class / chemical-property mappings
- **RECON3D** — genome-scale metabolic model (now with HMDB IDs from
  the Virtual Metabolic Human export)

Each is fetched on demand and cached locally.

In [ ]:
source(file.path(here::here(), "scripts/R/_utils.R"))
library(MetaProViz)
library(OmnipathR)

## MetalinksDB

In [ ]:
# What tables does MetalinksDB expose?
metalinksdb_tables()

In [ ]:
# The 'lr' table is the ligand-receptor edge list — small molecule on
# one side, protein receptor on the other.
ml_lr <- metalinksdb_table("lr")
dim(ml_lr)
head(ml_lr)

MetaProViz wraps MetalinksDB into a ready-to-enrich pathway-style table
via `metsigdb_metalinks()`:

In [ ]:
ml_pk <- metsigdb_metalinks()
head(ml_pk)

## HMDB

HMDB has dozens of fields per metabolite. List them, then pull a
focused slice.

In [ ]:
hmdb_metabolite_fields() |> head(20)

In [ ]:
# Just IDs and names — fast.
hmdb_ids <- hmdb_table(
    dataset = "metabolites",
    fields = c("accession", "name", "chemical_formula", "monisotopic_molecular_weight")
)
nrow(hmdb_ids)
head(hmdb_ids)

## RaMP-DB

RaMP unifies pathway memberships and chemical properties from KEGG,
Reactome, WikiPathways, HMDB, ChEBI, and LipidMaps.

In [ ]:
ramp_tables() |> head()

In [ ]:
# Pathway memberships in long form.
ramp_pathways <- ramp_table("source")
head(ramp_pathways)

## RECON3D

Useful when you want to reason about reactions, compartments, or
enzyme-metabolite relationships rather than pathway membership.

In [ ]:
recon_meta <- recon3d_metabolites()
head(recon_meta)

In [ ]:
recon_reactions <- recon3d_reactions()
head(recon_reactions)

## Compare PK coverage

Are the same metabolites covered by all of these? Use
`compare_pk()` to see the intersection / specificity.

In [ ]:
pk_overlap <- compare_pk(
    data = list(
        metalinks = as.data.frame(ml_pk),
        kegg      = as.data.frame(metsigdb_kegg())
    ),
    plot_name = "Metabolite coverage: MetalinksDB vs KEGG",
    save_plot = "svg",
    print_plot = TRUE,
    path = mp_results_dir("04_prior_knowledge")
)

## Connect to our dma table

`checkmatch_pk_to_data()` is the diagnostic that catches dimension or
ID-type mismatches early — it tells you how many features in the data
have at least one PK term, and how many PK terms have at least one
matching feature.

In [ ]:
dma_results <- readRDS(file.path(mp_results_dir(), "dma_results.rds"))
dma_table <- dma_results[["786-M1A_vs_HK2"]]$dma

# Build a feature metadata table with HMDB IDs (script 03 showed how;
# in real use this comes from the dataset's annotation file).
feature_meta <- dma_table |>
    dplyr::select(Metabolite) |>
    equivalent_id(metadata_info = c(InputID = "Metabolite"), from = "name")

In [ ]:
match_diag <- checkmatch_pk_to_data(
    data = feature_meta,
    input_pk = ml_pk,
    metadata_info = c(InputID = "HMDB", PriorID = "hmdb", grouping_variable = NULL)
)

names(match_diag)
match_diag$data_summary

In [ ]:
saveRDS(ml_pk, file.path(mp_results_dir(), "metalinks_pk.rds"))

**Recap.** Four metabolomics-relevant OmnipathR resources, each with a
single accessor. `compare_pk()` shows where they overlap;
`checkmatch_pk_to_data()` is the gate before enrichment in script 05.

# 5. Enrichment analysis

Two complementary approaches:

- **`standard_ora()`** — over-representation analysis: are the
  significantly altered metabolites enriched in any pathway? Fisher's
  exact test, the same engine `clusterProfiler` uses.
- **`cluster_pk()`** (new in v3.99.40) — first cluster pathway terms
  that share metabolites, then run ORA on representative clusters.
  Cuts redundancy in the output (e.g. you get one "TCA cycle"
  cluster instead of three near-duplicate KEGG/Reactome/Hallmark
  variants).

In [ ]:
source(file.path(here::here(), "scripts/R/_utils.R"))
library(MetaProViz)
library(OmnipathR)
library(dplyr)

dma_results <- readRDS(file.path(mp_results_dir(), "dma_results.rds"))

## Build a pathway resource — KEGG via MetaProViz

In [ ]:
kegg_pw <- metsigdb_kegg()
head(kegg_pw)

## Standard ORA per contrast

`standard_ora()` expects a table with `Metabolite` row names and
`t.val` / `p.adj` columns — exactly what `dma()` produces.

In [ ]:
ora_results <- list()

for (contrast in names(dma_results)) {
    dma_tab <- dma_results[[contrast]]$dma |>
        tibble::column_to_rownames("Metabolite")

    ora_results[[contrast]] <- standard_ora(
        data = dma_tab,
        metadata_info = c(
            pvalColumn = "p.adj",
            percentageColumn = "t.val",
            PathwayTerm = "term",
            PathwayFeature = "Metabolite"
        ),
        input_pathway = kegg_pw,
        pathway_name = "KEGG",
        cutoff_stat = 0.05,
        cutoff_percentage = 10,
        min_gssize = 3,
        max_gssize = 1000,
        save_table = "csv",
        path = mp_results_dir("05_enrichment")
    )
}

In [ ]:
ora_results[["786-M1A_vs_HK2"]]$ClusterGosummary |> head()

## Volcano-style PEA plot

`viz_volcano(plot_types = "PEA")` plots the enrichment results next to
the per-metabolite plot.

In [ ]:
viz_volcano(
    plot_types = "PEA",
    data = dma_results[["786-M1A_vs_HK2"]]$dma |> tibble::column_to_rownames("Metabolite"),
    data2 = ora_results[["786-M1A_vs_HK2"]]$ClusterGosummary,
    plot_name = "786-M1A vs HK2 — KEGG ORA",
    save_plot = "svg",
    print_plot = TRUE,
    path = mp_results_dir("05_enrichment")
)

## Reduce redundancy with `cluster_pk()`

Many pathway resources contain near-duplicate terms. `cluster_pk()`
clusters terms by their metabolite-set Jaccard similarity, then we
pick a representative per cluster before ORA.

In [ ]:
clustered_kegg <- cluster_pk(
    data = kegg_pw,
    metadata_info = c(term = "term", feature = "Metabolite"),
    cutoff_similarity = 0.5
)

names(clustered_kegg)
head(clustered_kegg$ClusteredPK)

Re-run ORA on the dereplicated PK and compare top hits.

In [ ]:
ora_clustered <- standard_ora(
    data = dma_results[["786-M1A_vs_HK2"]]$dma |> tibble::column_to_rownames("Metabolite"),
    metadata_info = c(
        pvalColumn = "p.adj",
        percentageColumn = "t.val",
        PathwayTerm = "term",
        PathwayFeature = "Metabolite"
    ),
    input_pathway = clustered_kegg$ClusteredPK,
    pathway_name = "KEGG_clustered",
    cutoff_stat = 0.05,
    cutoff_percentage = 10,
    save_table = "csv",
    path = mp_results_dir("05_enrichment")
)

ora_clustered$ClusterGosummary |> head()

## Chemical-class enrichment (added v2.1.4)

The same ORA machinery runs against chemical classes — useful when
pathway annotation is sparse but compound-class info is rich.

In [ ]:
chem_pk <- metsigdb_chemicalclass()
head(chem_pk)

In [ ]:
ora_chem <- standard_ora(
    data = dma_results[["786-M1A_vs_HK2"]]$dma |> tibble::column_to_rownames("Metabolite"),
    metadata_info = c(
        pvalColumn = "p.adj",
        percentageColumn = "t.val",
        PathwayTerm = "term",
        PathwayFeature = "Metabolite"
    ),
    input_pathway = chem_pk,
    pathway_name = "ChemicalClass",
    cutoff_stat = 0.05,
    cutoff_percentage = 10,
    save_table = "csv",
    path = mp_results_dir("05_enrichment")
)

ora_chem$ClusterGosummary |> head()

In [ ]:
saveRDS(ora_results, file.path(mp_results_dir(), "ora_kegg.rds"))

**Recap.** Three enrichment angles on the same dma table: raw KEGG ORA,
de-duplicated KEGG ORA via `cluster_pk()`, and chemical-class ORA. The
next script visualises one of these as a small network.

# 6. Network visualization with `viz_graph()`

`viz_graph()` (new in MetaProViz 3.99.40) renders a node-link graph
from a long-format edge table. We use it to draw a small ligand-receptor
subnetwork around the metabolites that drove the top KEGG cluster from
script 05.

In [ ]:
source(file.path(here::here(), "scripts/R/_utils.R"))
library(MetaProViz)
library(OmnipathR)
library(dplyr)

## Pick a focus set of metabolites

In [ ]:
dma_results <- readRDS(file.path(mp_results_dir(), "dma_results.rds"))

top_metabolites <- dma_results[["786-M1A_vs_HK2"]]$dma |>
    dplyr::arrange(p.adj) |>
    dplyr::slice_head(n = 15) |>
    dplyr::pull(Metabolite)

top_metabolites

## Slice MetalinksDB to those metabolites

We pull metabolite ↔ receptor / transporter edges, then filter to our
top-15 set.

In [ ]:
ml_lr <- metalinksdb_table("lr")

ml_subset <- ml_lr |>
    dplyr::filter(metabolite_name %in% top_metabolites) |>
    dplyr::distinct(metabolite_name, gene_symbol, type)

dim(ml_subset)
head(ml_subset)

## Draw the graph

In [ ]:
viz_graph(
    data = ml_subset,
    metadata_info = c(source = "metabolite_name", target = "gene_symbol", type = "type"),
    plot_name = "Top differential metabolites — MetalinksDB neighbours",
    save_plot = "svg",
    print_plot = TRUE,
    path = mp_results_dir("06_network")
)

## Optional: a COSMOS PKN teaser

OmnipathR also ships builders that compose the full COSMOS prior
knowledge network — multi-layer (transporters, receptors, allosteric,
enzyme–metabolite, signalling, regulation). We won't run COSMOS itself,
but here's the entry point — it's the same data the Python segment
explores from a different angle.

In [ ]:
# Just print the function, don't run — full PKN is heavy.
?cosmos_pkn

**End of R section.**

Next: switch kernel to Python and continue with `notebooks/metabo_python.ipynb`.